In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

import plotly as ply
import json

import requests

import warnings
warnings.filterwarnings("ignore")

In [ ]:
def get_property_summary(ain: str):
    #print(ain)
    url = 'https://portal.assessor.lacounty.gov/api/parceldetail?ain='+ain
    page = requests.get(url)

    # Store JSON data in API_Data
    try:
        API_Data = page.json()
    except json.JSONDecodeError as e:
        print("Invalid JSON:", e)
        return np.nan,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan

    return ain, API_Data['Parcel']['SitusStreet'],API_Data['Parcel']['SitusZipCode'],API_Data['Parcel']['TaxStatus'],API_Data['Parcel']['TaxDefaultedYear'],API_Data['Parcel']['ParcelStatus'],API_Data['Parcel']['NumOfUnits'],API_Data['Parcel']['YearBuilt'],API_Data['Parcel']['SqftMain'],API_Data['Parcel']['SqftLot'],API_Data['Parcel']['NumOfBeds'],API_Data['Parcel']['NumOfBaths'],API_Data['Parcel']['RollPreparation_LandValue'],API_Data['Parcel']['RollPreparation_ImpValue']

In [ ]:
def get_ownership_history(ain:str):
    #print(ain)
    url = 'https://portal.assessor.lacounty.gov/api/parcel_ownershiphistory?ain='+ain
    page = requests.get(url)

    # Store JSON data in API_Data
    try:
        history = page.json()
    except json.JSONDecodeError as e:
        print("Invalid JSON:", e)
        return np.nan,np.nan,np.nan

    return ain, history['Parcel_OwnershipHistory'][0]['RecordingDate'],history['Parcel_OwnershipHistory'][0]['DocumentNumber'],history['Parcel_OwnershipHistory'][0]['DocumentReasonCodeDesc']

In [ ]:
filename = '/content/drive/MyDrive/Big Data/Properties/LA tax-default 04-2024 organized .xlsx'

data = pd.read_excel(filename, sheet_name='raw data')
data.columns = ['Item','AIN','Min_bid','Improv','NSB_number','Location','Address']
data = data[~data['Address'].str.startswith('VAC')]
data['Address'] = data['Address'].str.replace('_x000D_', ' ')
data['AIN'] = data['AIN'].str.replace("-", "")
data['Improv'] = data['Improv'].fillna('N')
data.head()

,Item,AIN,Min_bid,Improv,NSB_number,Location,Address
0,448,3023057012,33866,Y,522,CITY-PALMDALE,37616 NEWBURY PL PALMDALE CA\n93552-3937
8,673,3069012019,6908,Y,522,COUNTY OF LOS ANGELES,40614 154TH ST E LAKE LOS\nANGELES CA 93535-7028
10,893,3131025001,79913,Y,522,CITY-LANCASTER,44058 FIG AVE LANCASTER CA\n93534-4437
15,1750,5013001011,6166,Y,536,COUNTY OF LOS ANGELES,5118 ANGELES VISTA BLVD LOS\nANGELES CA 90043-...
16,1837,5106011011,4593,Y,536,CITY-LOS ANGELES,1608 E 45TH ST LOS ANGELES CA\n90011-4312


In [ ]:
cols1 = ['AIN','SitusStreet','SitusZipCode','TaxStatus',
         'TaxDefaultedYear','ParcelStatus','NumOfUnits',
         'YearBuilt','SqftMain','SqftLot',
         'NumOfBeds','NumOfBaths','RollPreparation_LandValue',
         'RollPreparation_ImpValue']

res1 = data.AIN.apply(get_property_summary)
df1 = pd.DataFrame([x for x in res1], columns=cols1)

In [ ]:
df1[df1.TaxStatus=='CURRENT'].to_csv('/content/drive/MyDrive/Big Data/Properties/current_tax_status.csv')
print(len(df1[df1.TaxStatus=='CURRENT']))
df1[df1.TaxStatus=='CURRENT'].head(20)

74


,AIN,SitusStreet,SitusZipCode,TaxStatus,TaxDefaultedYear,ParcelStatus,NumOfUnits,YearBuilt,SqftMain,SqftLot,NumOfBeds,NumOfBaths,RollPreparation_LandValue,RollPreparation_ImpValue
0,3023057012,37616 NEWBURY PL,93552-3937,CURRENT,,ACTIVE,1,1990,1600,7217,3,2,44199,132496
1,3069012019,40614 154TH ST E,93535-7028,CURRENT,,ACTIVE,1,1985,1402,41803,3,2,23871,283385
2,3131025001,44058 FIG AVE,93534-4437,CURRENT,,ACTIVE,1,1952,1416,6257,3,2,95880,288660
3,5013001011,5118 ANGELES VISTA BLVD,90043-1749,CURRENT,,ACTIVE,1,1947,2016,6778,2,3,48510,47772
5,6023015021,743 E 76TH ST,90001-2803,CURRENT,,ACTIVE,1,1925,962,5105,2,1,200964,50235
14,2354012061,"11531 RIVERSIDE DR, NO 108",91602-1088,CURRENT,,ACTIVE,1,1990,1198,20914,2,2,253345,197340
15,2619024007,13535 PINNEY ST,91331-2949,CURRENT,,ACTIVE,1,1947,1022,7241,3,1,19331,38191
19,3069008019,40545 154TH ST E,93535-7027,CURRENT,,ACTIVE,1,1989,1134,39517,3,2,17930,71741
24,3145013006,167 E AVENUE F,93535-6411,CURRENT,,ACTIVE,0,1948,96,7140,1,0,2795,71
26,3264010002,48059 93RD ST W,93536-9331,CURRENT,,ACTIVE,1,1986,1348,108676,3,2,32651,97968


In [ ]:
cols2 = ['AIN','RecordingDate','DocumentNumber','DocumentReasonCodeDesc']

res2 = data.AIN.apply(get_ownership_history)
df2 = pd.DataFrame([x for x in res2], columns=cols2)

In [ ]:
merged_df = pd.merge(data, df1, on="AIN")
merged_df = pd.merge(merged_df, df2, on="AIN")

merged_df.head()

,Item,AIN,Min_bid,Improv,NSB_number,Location,Address,SitusStreet,SitusZipCode,TaxStatus,...,YearBuilt,SqftMain,SqftLot,NumOfBeds,NumOfBaths,RollPreparation_LandValue,RollPreparation_ImpValue,RecordingDate,DocumentNumber,DocumentReasonCodeDesc
0,448,3023057012,33866,Y,522,CITY-PALMDALE,37616 NEWBURY PL PALMDALE CA\n93552-3937,37616 NEWBURY PL,93552-3937,CURRENT,...,1990,1600,7217,3,2,44199,132496,03/26/2024,0191595,Joint Tenancy Transfer (SEC. 62F or 65C)
1,673,3069012019,6908,Y,522,COUNTY OF LOS ANGELES,40614 154TH ST E LAKE LOS\nANGELES CA 93535-7028,40614 154TH ST E,93535-7028,CURRENT,...,1985,1402,41803,3,2,23871,283385,06/22/1988,0989843,Interspousal Transfer (SEC. 63)
2,893,3131025001,79913,Y,522,CITY-LANCASTER,44058 FIG AVE LANCASTER CA\n93534-4437,44058 FIG AVE,93534-4437,CURRENT,...,1952,1416,6257,3,2,95880,288660,12/04/2023,0838097,Good Transfer - Typical Change in Ownership
3,1750,5013001011,6166,Y,536,COUNTY OF LOS ANGELES,5118 ANGELES VISTA BLVD LOS\nANGELES CA 90043-...,5118 ANGELES VISTA BLVD,90043-1749,CURRENT,...,1947,2016,6778,2,3,48510,47772,12/13/2002,3059957,Perfection of Title (SEC. 62B)
4,1837,5106011011,4593,Y,536,CITY-LOS ANGELES,1608 E 45TH ST LOS ANGELES CA\n90011-4312,1608 E 45TH ST,90011-4312,DEFAULTED,...,1922,1616,5200,4,2,15651,12473,08/16/1985,0951775,Partial Interest Transfer - Complete Own-216


In [ ]:
merged_df[merged_df.TaxStatus=='CURRENT'].head(48)

,Item,AIN,Min_bid,Improv,NSB_number,Location,Address,SitusStreet,SitusZipCode,TaxStatus,...,YearBuilt,SqftMain,SqftLot,NumOfBeds,NumOfBaths,RollPreparation_LandValue,RollPreparation_ImpValue,RecordingDate,DocumentNumber,DocumentReasonCodeDesc
1,673,3069012019,6908,Y,522,COUNTY OF LOS ANGELES,40614 154TH ST E LAKE LOS\nANGELES CA 93535-7028,40614 154TH ST E,93535-7028,CURRENT,...,1985,1402,41803,3,2,23403,277829,06/22/1988,0989843,Interspousal Transfer (SEC. 63)
2,893,3131025001,79913,Y,522,CITY-LANCASTER,44058 FIG AVE LANCASTER CA\n93534-4437,44058 FIG AVE,93534-4437,CURRENT,...,1952,1416,6257,3,2,94000,283000,12/04/2023,0838097,Good Transfer - Typical Change in Ownership
3,1750,5013001011,6166,Y,536,COUNTY OF LOS ANGELES,5118 ANGELES VISTA BLVD LOS\nANGELES CA 90043-...,5118 ANGELES VISTA BLVD,90043-1749,CURRENT,...,1947,2016,6778,2,3,47559,46836,12/13/2002,3059957,Perfection of Title (SEC. 62B)
7,2876,7319008029,57687,Y,509,CITY-CARSON,17544 HARWICK CT CARSON CA 90746-1618,17544 HARWICK CT,90746-1618,CURRENT,...,1966,1475,4745,3,2,181086,181086,01/15/2016,0052183,Transfers Not Changing % Interests (SEC. 62A)
8,3032,8059028038,34584,Y,607,CITY-SANTA FE SP,14020 MARQUARDT AVE SANTA FE\nSPRINGS CA 90670...,14020 MARQUARDT AVE,90670-5019,CURRENT,...,1979,4036,113726,0,0,133967,455519,02/02/2006,0253723,Good Transfer - Typical Change in Ownership
14,3506,2354012061,4108,Y,536,CITY-LOS ANGELES,11531 RIVERSIDE DR NO 108 LOS ANGELES CA 91602...,"11531 RIVERSIDE DR, NO 108",91602-1088,CURRENT,...,1990,1198,20914,2,2,248378,193471,11/19/2014,1241801,Good Transfer - Typical Change in Ownership
15,3514,2619024007,12152,Y,536,CITY-LOS ANGELES,13535 PINNEY ST LOS ANGELES CA\n91331-2949,13535 PINNEY ST,91331-2949,CURRENT,...,1947,1022,7241,3,1,18952,37443,11/19/1991,9999999,Parent-Child Transfer (Prop. 58)
19,3556,3069008019,29973,Y,522,COUNTY OF LOS ANGELES,40545 154TH ST E LANCASTER CA\n93535-7027,40545 154TH ST E,93535-7027,CURRENT,...,1989,1134,39517,3,2,17579,70335,11/22/2010,1692646,Non-reappraisable Trust Transfer (SEC. 62D)
24,3589,3145013006,5889,Y,522,COUNTY OF LOS ANGELES,167 E AVENUE F LANCASTER CA\n93535-6411,167 E AVENUE F,93535-6411,CURRENT,...,1948,96,7140,1,0,2741,70,09/25/2015,1194322,Partial Interest Transfer - Complete Own-216
26,3633,3264010002,28860,Y,522,COUNTY OF LOS ANGELES,48059 93RD ST W LANCASTER CA\n93536-9331,48059 93RD ST W,93536-9331,CURRENT,...,1986,1348,108676,3,2,21153,87935,09/15/1995,1511823,Good Transfer - Typical Change in Ownership
